In [1]:
from openai_harmony import (
    Author,
    Conversation,
    DeveloperContent,
    HarmonyEncodingName,
    Message,
    Role,
    SystemContent,
    ToolDescription,
    load_harmony_encoding,
    ReasoningEffort
)
 
encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
 
system_message = (
    SystemContent.new()
        .with_reasoning_effort(ReasoningEffort.HIGH)
        .with_conversation_start_date("2025-06-28")
)
 
developer_message = (
    DeveloperContent.new()
        .with_instructions("Always respond in riddles")
        .with_function_tools(
            [
                ToolDescription.new(
                    "get_current_weather",
                    "Gets the current weather in the provided location.",
                    parameters={
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            },
                            "format": {
                                "type": "string",
                                "enum": ["celsius", "fahrenheit"],
                                "default": "celsius",
                            },
                        },
                        "required": ["location"],
                    },
                ),
            ]
	)
)
 
convo = Conversation.from_messages(
    [
        Message.from_role_and_content(Role.SYSTEM, system_message),
        Message.from_role_and_content(Role.DEVELOPER, developer_message),
        Message.from_role_and_content(Role.USER, "What is the weather in Tokyo?"),
        Message.from_role_and_content(
            Role.ASSISTANT,
            'User asks: "What is the weather in Tokyo?" We need to use get_weather tool.',
        ).with_channel("analysis"),
        Message.from_role_and_content(Role.ASSISTANT, '{"location": "Tokyo"}')
        .with_channel("commentary")
        .with_recipient("functions.get_weather")
        .with_content_type("<|constrain|> json"),
        Message.from_author_and_content(
            Author.new(Role.TOOL, "functions.lookup_weather"),
            '{ "temperature": 20, "sunny": true }',
        ).with_channel("commentary"),
    ]
)

convo_dict = convo.to_dict()

print(convo.to_json())
 
tokens = encoding.render_conversation_for_completion(convo, Role.ASSISTANT)
 
print(tokens)
 
# After receiving a token response
# Do not pass in the stop token

# parsed_response = encoding.parse_messages_from_completion_tokens(tokens, Role.ASSISTANT)

{"messages": [{"role": "system", "name": null, "content": [{"model_identity": "You are ChatGPT, a large language model trained by OpenAI.", "reasoning_effort": "High", "conversation_start_date": "2025-06-28", "knowledge_cutoff": "2024-06", "channel_config": {"valid_channels": ["analysis", "commentary", "final"], "channel_required": true}, "type": "system_content"}]}, {"role": "developer", "name": null, "content": [{"instructions": "Always respond in riddles", "tools": {"functions": {"name": "functions", "tools": [{"name": "get_current_weather", "description": "Gets the current weather in the provided location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}, "format": {"type": "string", "enum": ["celsius", "fahrenheit"], "default": "celsius"}}, "required": ["location"]}}]}}, "type": "developer_content"}]}, {"role": "user", "name": null, "content": [{"type": "text", "text": "What is the weather

In [3]:
from openai import AsyncClient
import os

lightning_client = AsyncClient(
    base_url=os.getenv("LIGHTNING_SERVER_BASE_URL"),
    api_key=os.getenv("LIGHTNING_STUDIO_API"),
)

completion = await lightning_client.chat.completions.create(
    model="lightning-ai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": [{"type": "text", "text": "Tell me about yourself"}]
        },
    ],
)
print(completion)

ChatCompletion(id='chatcmpl-3252bd6712834c9d9567dd5556bcf9aa', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I’m ChatGPT—an AI language model created by OpenAI. I’m built on the GPT‑4 architecture, which means I’m trained on a vast mix of text from books, articles, websites, and other sources up until mid‑2023. That training lets me:\n\n- **Understand and generate natural language**: I can read your prompt, grasp intent, and craft replies that sound conversational.\n- **Answer a wide range of questions**: From science and history to cooking tips and coding help, I pull in general knowledge from my training data (but I don’t browse the web in real time).\n- **Assist with creativity and productivity**: I can help brainstorm ideas, draft emails, write poems, or outline projects.\n- **Learn from context**: While I don’t retain personal data between sessions, I can keep track of what’s happening during our conversation to stay coherent.

In [5]:
from IPython.display import Markdown

In [9]:
Markdown(completion.choices[0].message.reasoning_content)

The user says: "Tell me about yourself". We have to respond as ChatGPT. We should produce a friendly answer discussing we are an AI model. It's an OpenAI ChatGPT. We can talk about training data, language capabilities. Possibly mention not a human. It's fine. Let's produce a moderate length.

In [7]:
Markdown(completion.choices[0].message.content)

I’m ChatGPT—an AI language model created by OpenAI. I’m built on the GPT‑4 architecture, which means I’m trained on a vast mix of text from books, articles, websites, and other sources up until mid‑2023. That training lets me:

- **Understand and generate natural language**: I can read your prompt, grasp intent, and craft replies that sound conversational.
- **Answer a wide range of questions**: From science and history to cooking tips and coding help, I pull in general knowledge from my training data (but I don’t browse the web in real time).
- **Assist with creativity and productivity**: I can help brainstorm ideas, draft emails, write poems, or outline projects.
- **Learn from context**: While I don’t retain personal data between sessions, I can keep track of what’s happening during our conversation to stay coherent.

I don’t have feelings, consciousness, or personal experiences—I’m a tool that processes patterns in text and produces responses that are statistically likely to be helpful. My goal is to be useful, safe, and engaging, so feel free to ask me anything or let me know how I can assist you next!